# Introduction to GPU Programming with Triton

This notebook is an **introduction to Triton**, a Python-like language for writing high-performance GPU kernels, and how it relates to **CUDA programming concepts** like threads and blocks.  

---

## CUDA Basics

When programming GPUs with **CUDA**, there are a few key concepts to understand:

### Threads
- The smallest unit of execution on a GPU.
- Each thread computes a single part of a problem, like one element of a matrix.
- Think of a thread as a “worker” that performs a tiny task.

### Blocks
- Threads are grouped into **blocks**.
- Threads in a block can share **fast shared memory** and synchronize with each other.
- Each block computes a portion of the output, like a small tile of a matrix.

### Grids
- Blocks are organized into a **grid** to cover the entire problem.
- Together, all blocks compute the full output.


## What is a CUDA Thread?

In CUDA programming:

- A **thread** is the **smallest unit of execution on the GPU**.
- Each thread executes the **same kernel code**, but usually operates on **different data**.

**Thread indexing:**

- `threadIdx` — the thread’s index **within its block**.
- `blockIdx` — the block’s index **within the grid**.
- `blockDim` — the number of threads **per block**.
- To get the **global index** of a thread (across all blocks):


In [1]:
from numba import cuda, float32
cuda.float32 = float32

In [2]:
import numpy as np
from numba import cuda


@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Get the global thread index
    idx = cuda.grid(1)

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)  # decreasing values
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads and blocks
threads_per_block = 8
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/cudadrv/devicearray.py:887: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## What is a CUDA Block?

In CUDA programming:

- A **block** is a group of threads that execute together on a single **Streaming Multiprocessor (SM)** on the GPU.
- Threads in a block can:
  - Share **fast shared memory**.
  - Synchronize with each other using `cuda.syncthreads()`.
  
- Blocks are organized into a **grid** to cover larger datasets.

**Thread hierarchy:**

1. **Thread:** smallest unit of execution.
2. **Block:** a group of threads that can communicate via shared memory.
3. **Grid:** a group of blocks.

**Thread indexing within a block:**

- `threadIdx.x` — thread’s index **inside the block**.
- `blockIdx.x` — block’s index **inside the grid**.
- `blockDim.x` — number of threads **per block**.

**Global thread index formula:**


```global_idx = threadIdx.x + blockIdx.x * blockDim.x```



Each block is independent; threads in different blocks **cannot directly share shared memory**.


In [3]:
import numpy as np
from numba import cuda


# CUDA kernel for vector addition
@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Thread index inside block
    tx = cuda.threadIdx.x

    # Block index inside grid
    bx = cuda.blockIdx.x

    # Number of threads per block
    bdim = cuda.blockDim.x

    # Compute global thread index
    idx = tx + bx * bdim

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads per block and number of blocks
threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]
Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/cudadrv/devicearray.py:887: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## Why Use Blocks in CUDA?

Using blocks is essential for **performance and scalability**:

1. **Shared Memory:**
   - Threads within the same block can share **fast shared memory**.
   - This avoids repeatedly reading the same data from **slower global memory**.

2. **Thread Cooperation:**
   - Threads in a block can **synchronize** using `cuda.syncthreads()`.
   - Useful for reductions, scans, or any operation where threads depend on each other.

3. **Tiling / Block-wise Computation:**
   - Large datasets can be **split into blocks (tiles)**.
   - Each block loads a chunk of data into shared memory, computes results locally, and writes back.
   - This **reduces global memory accesses** and improves performance.

We’ll demonstrate this with **tiled vector addition**, where each block loads its portion of two vectors into shared memory, adds them, and writes back the results.

## Tiled Vector Addition

Imagine we have:

- Vector A = [0, 1, 2, 3, 4, 5, 6, 7]  
- Vector B = [8, 7, 6, 5, 4, 3, 2, 1]  
- Threads per block = 4 → 2 blocks

---

### Step 1: Divide into Blocks (Tiles)

| Block | Thread IDs (tx) | Global Index (idx) |
|-------|----------------|------------------|
| 0     | 0,1,2,3       | 0,1,2,3         |
| 1     | 0,1,2,3       | 4,5,6,7         |

---

### Step 2: Load Tile into Shared Memory
```
Block 0 Shared Memory (sA, sB):
sA = [0,1,2,3]
sB = [8,7,6,5]

Block 1 Shared Memory (sA, sB):
sA = [4,5,6,7]
sB = [4,3,2,1]
```


- Each thread loads **one element** into shared memory.
- Threads in the block synchronize with `cuda.syncthreads()`.

---

### Step 3: Compute in Shared Memory
```
Block 0:
Thread 0: sA[0] + sB[0] = 0 + 8 = 8
Thread 1: sA[1] + sB[1] = 1 + 7 = 8
Thread 2: sA[2] + sB[2] = 2 + 6 = 8
Thread 3: sA[3] + sB[3] = 3 + 5 = 8

Block 1:
Thread 0: sA[0] + sB[0] = 4 + 4 = 8
Thread 1: sA[1] + sB[1] = 5 + 3 = 8
Thread 2: sA[2] + sB[2] = 6 + 2 = 8
Thread 3: sA[3] + sB[3] = 7 + 1 = 8
```

---

### Step 4: Write Back to Global Memory

- Each thread writes its computed value from **shared memory** to **global memory**.
- Multiple blocks work independently in parallel.


In [4]:
import numpy as np
from numba import cuda

MAX_SIZE = 32  # choose a safe upper bound for threads per block


# CUDA kernel: Tiled vector addition using "static" shared memory with flexible size
@cuda.jit
def vector_add_tiled(A: np.ndarray, B: np.ndarray, C: np.ndarray, sA_size: int, sB_size: int):
    # Declare max possible shared memory size
    sA = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)
    sB = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    idx = tx + bx * bdim

    # Only use the portion of shared memory according to passed sizes
    if idx < C.size and tx < sA_size:
        sA[tx] = A[idx]
    if idx < C.size and tx < sB_size:
        sB[tx] = B[idx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size and tx < sB_size:
        sA[tx] = sA[tx] + sB[tx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size:
        C[idx] = sA[tx]


# Data
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel with sizes for sA and sB
vector_add_tiled[blocks_per_grid, threads_per_block](A, B, C, threads_per_block, threads_per_block)

print("Vector C:", C)

Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/raibotics/MIPT/env/lib/python3.12/site-packages/numba/cuda/cudadrv/devicearray.py:887: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Triton Basics

Triton simplifies GPU programming by **removing much of the low-level thread/block management**.  

## Program
- In Triton, a **program** is like a “super-thread” that can operate on **a whole tile of data**.
- A program handles multiple elements at once using **vectorized operations**.
- Triton automatically maps these programs onto GPU threads and warps.

## Threads vs Programs
| Concept | CUDA | Triton |
|---------|------|--------|
| Smallest unit | Thread (1 element) | Program (1 tile of elements) |
| Shared memory | Threads in a block | Program uses fast memory implicitly |
| Mapping | ThreadIdx / BlockIdx | tl.program_id() |

## Tiled Vector Addition Using Triton

In Triton:

- A **kernel** is written as a Python function decorated with `@triton.jit`.
- Threads are organized in **blocks**, called **program IDs**.
- We can load **tiles of data into shared memory (using Triton’s `tl.load`)** for faster computation.
- Each thread (or “program”) handles a portion of the tile.

We will implement **vector addition C = A + B** using **tiled blocks**.

In [2]:
import torch
print("Version:", torch.__version__)
print("Is CUDA available?:", torch.cuda.is_available())

# Тест тензора
try:
    x = torch.ones(1).cuda()
    print("Success! GPU is working.")
except Exception as e:
    print("Still failing:", e)

Version: 2.7.1+cu118
Is CUDA available?: True
Success! GPU is working.


In [1]:
import torch
import triton
import triton.language as tl


# Triton kernel: vector addition with tiling
@triton.jit
def vector_add_tiled_kernel(
        A_ptr: torch.Tensor,
        B_ptr: torch.Tensor,
        C_ptr: torch.Tensor,
        N: tl.constexpr,
        BLOCK_SIZE: tl.constexpr
):
    # program ID = block index
    pid = tl.program_id(0)

    # compute start index of this block
    start = pid * BLOCK_SIZE

    # compute offsets for threads in the block
    offsets = start + tl.arange(0, BLOCK_SIZE)

    # Mask for threads that go out of bounds
    mask = offsets < N

    # Load from global memory
    a = tl.load(A_ptr + offsets, mask=mask)
    b = tl.load(B_ptr + offsets, mask=mask)

    # Compute addition
    c = a + b

    # Write back to global memory
    tl.store(C_ptr + offsets, c, mask=mask)


# Example vectors
N = 16
BLOCK_SIZE = 4
A = torch.arange(N, dtype=torch.float32, device='cuda')
B = torch.arange(N, 0, -1, dtype=torch.float32, device='cuda')
C = torch.zeros(N, dtype=torch.float32, device='cuda')

print("Vector A:", A)
print("Vector B:", B)

# Launch kernel
grid = (triton.cdiv(N, BLOCK_SIZE),)
vector_add_tiled_kernel[grid](A, B, C, N, BLOCK_SIZE=BLOCK_SIZE)

print("Result Vector C:", C)

Vector A: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')
Vector B: tensor([16., 15., 14., 13., 12., 11., 10.,  9.,  8.,  7.,  6.,  5.,  4.,  3.,
         2.,  1.], device='cuda:0')
Result Vector C: tensor([16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 16.], device='cuda:0')


## Why Tiled Vector Addition in Triton May Not Give Performance Boost

1. **Vector addition is memory-bound**:  
   - Each element is **read once from A and B, written once to C**.  
   - GPU arithmetic is cheap; most of the time is spent moving data.  

2. **Tiling only helps when data is reused**:  
   - In vector addition, each element is used **exactly once**.  
   - Loading tiles into shared memory (registers) **does not reduce global memory traffic**, because we still read each element once.  

3. **Overhead of tiling**:  
   - Extra instructions to compute offsets and masks may **slightly increase runtime** for small vectors.  

## Triton Matrix Multiplication Using Tiling


We are computing **C = A @ B**, where:

- A: (M x K)  
- B: (K x N)  
- C: (M x N)  

We use **tiled blocks** to exploit GPU parallelism and fast memory.

---

## 1. Block-level parallelism

- Each block computes a **tile of C** with size `BLOCK_M x BLOCK_N`.
- **Program IDs** identify each block:

```
pid_m = tl.program_id(0) # block row index
pid_n = tl.program_id(1) # block column index
```

- Block start indices:

```
m_start = pid_m * BLOCK_M
n_start = pid_n * BLOCK_N
```

- Example:  

```
BLOCK_M = 8, BLOCK_N = 8
pid_m=0, pid_n=0 -> computes C[0:8, 0:8]
pid_m=1, pid_n=0 -> computes C[8:16, 0:8]

```

---

## 2. Tiles and shared registers

- **accumulator**: each block creates a local `BLOCK_M x BLOCK_N` matrix in **registers**:


```acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)```


- This holds the **partial sum** of the block’s tile of C.
- Each element of the tile is computed by **multiple multiply-adds** over the K dimension.

---

## 3. Looping over K dimension in tiles

- The K dimension is usually too large to load fully into fast memory.
- Loop over K in **BLOCK_K chunks**:


```
for k_start in range(0, K, BLOCK_K):
    load A_tile: shape (BLOCK_M x BLOCK_K)
    load B_tile: shape (BLOCK_K x BLOCK_N)
    acc += A_tile @ B_tile
```

- Each tile is loaded from **global memory** into registers.  
- **Multiply-accumulate (tl.dot)** is performed in fast memory, **reducing global memory accesses**.

---

## 4. Offsets and masking

- Compute **global indices** for each tile:


```
offs_m = m_start + tl.arange(0, BLOCK_M)
offs_n = n_start + tl.arange(0, BLOCK_N)
offs_k = k_start + tl.arange(0, BLOCK_K)
```

- `tl.load` uses a **mask** to avoid out-of-bounds reads:


```mask=(offs_m[:, None] < M) & (offs_k[None, :] < K)```


- This ensures safety for the **last blocks** if M, N, or K are not multiples of tile size.

---

## 5. Multiply-accumulate (tl.dot)

- Compute:


```acc += tl.dot(A_tile, B_tile)```


- Each element in the tile of C is the **sum of products** along the K dimension.  
- After looping over all K tiles, `acc` contains the **final values** for this block’s C tile.

---

## 6. Write back to global memory

- After all K tiles are processed:


```tl.store(C_ptr + offs_m[:, None] * N + offs_n[None, :], acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))```


- Each block writes its tile of C to the correct location in global memory.  
- Multiple blocks write **independently in parallel**, covering the entire matrix.

---

## 7. Summary of Benefits

- **Tiling reduces global memory traffic**: each A and B element is loaded **once per tile** but used many times.  
- **Registers hold temporary results**, avoiding slow global memory for intermediate computations.  
- **Blocks operate independently** → full GPU parallelism.  
- **Masking ensures safety** for non-multiple dimensions.  

In [3]:
import torch
import triton
import triton.language as tl


@triton.jit
def matmul_kernel(
        A_ptr: torch.Tensor,  # tensor A
        B_ptr: torch.Tensor,  # tensor B
        C_ptr: torch.Tensor,  # tensor C
        M: tl.constexpr,  # number of rows in A/C
        N: tl.constexpr,  # number of columns in B/C
        K: tl.constexpr,  # number of columns in A / rows in B
        BLOCK_M: tl.constexpr,  # tile height
        BLOCK_N: tl.constexpr,  # tile width
        BLOCK_K: tl.constexpr  # tile depth along K
):
    ...
# Example usage
M, N, K = 16, 16, 16
BLOCK_M, BLOCK_N, BLOCK_K = 8, 8, 16  # BLOCK_K >= 16 required for tl.dot

A = torch.randn((M, K), dtype=torch.float32, device='cuda')
B = torch.randn((K, N), dtype=torch.float32, device='cuda')
C = torch.zeros((M, N), dtype=torch.float32, device='cuda')

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
matmul_kernel[grid](A, B, C, M, N, K, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K)

print("Matrix C:\n", C)

Matrix C:
 tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.

In [4]:
@triton.jit
def _seeded_dropout(
        x_ptr: torch.Tensor,
        output_ptr: torch.Tensor,
        n_elements: int,
        p: float,
        seed: int,
        BLOCK_SIZE: tl.constexpr,
):
    ...


def seeded_dropout(x: torch.Tensor, p: float, seed: int) -> torch.Tensor:
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    _seeded_dropout[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


x = torch.randn(size=(10,)).cuda()
# Compare this to the baseline - dropout mask is never instantiated!
output = seeded_dropout(x, p=0.5, seed=123)

print("Input: ", x)
print("Out: ", output)

Input:  tensor([ 0.3812,  0.7369, -1.1369, -0.9142, -0.9603,  1.2243, -0.5486, -0.1124,
        -0.3742,  0.3730], device='cuda:0')
Out:  tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')


# Homework

Реализуйте операторы на GELU. каждый оператор

# Task 1: Row-wise Matrix Sum

**Problem:**  

Implement a Triton kernel that computes **the sum of each row** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `row_sums` of shape `(M,)`  
- Hint:  
  - Use **tiles along the N dimension**  
  - Accumulate the sum in **registers**  
  - Handle partial tiles with **masking**

In [6]:
import torch
import triton
import triton.language as tl

@triton.jit
def row_sum_kernel(A_ptr: torch.Tensor,
                   row_sums_ptr: torch.Tensor,
                   M: int, N: int,
                   BLOCK_N: tl.constexpr):
    row = tl.program_id(0)

    if BLOCK_N <= 1:
        offs = tl.arange(0, 1)
    elif BLOCK_N <= 2:
        offs = tl.arange(0, 2)
    elif BLOCK_N <= 4:
        offs = tl.arange(0, 4)
    elif BLOCK_N <= 8:
        offs = tl.arange(0, 8)
    elif BLOCK_N <= 16:
        offs = tl.arange(0, 16)
    elif BLOCK_N <= 32:
        offs = tl.arange(0, 32)
    elif BLOCK_N <= 64:
        offs = tl.arange(0, 64)
    else:
        offs = tl.arange(0, 128)

    acc = 0.0

    for start_n in range(0, N, BLOCK_N):
        cols = start_n + offs
        mask = (row < M) & (offs < BLOCK_N) & (cols < N)
        vals = tl.load(A_ptr + row * N + cols, mask=mask, other=0.0)
        acc += tl.sum(vals, axis=0)

    if row < M:
        tl.store(row_sums_ptr + row, acc)

def run_kernel(A, BLOCK_N):
    M, N = A.shape
    row_sums = torch.zeros(M, device=A.device, dtype=A.dtype)
    
    # Grid из M программ (по одной на каждую строку)
    grid = (M,)
    
    row_sum_kernel[grid](
        A, row_sums, 
        M, N, 
        BLOCK_N=BLOCK_N
    )
    return row_sums


def reference(A):
    return torch.sum(A, dim=1)


# --- Test 1: small fixed matrix ---
M, N, BLOCK_N = 4, 8, 4
A = torch.arange(M * N, dtype=torch.float32, device='cuda').reshape(M, N)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Matrix A:\n", A)
print("Kernel row sums:\n", row_sums)
print("Reference row sums:\n", row_sums_ref)
print("Match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 2: random matrix ---
M, N, BLOCK_N = 7, 13, 5  # not divisible by BLOCK_N
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Random test, match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_N = 64, 128, 32
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Large test, match?", torch.allclose(row_sums, row_sums_ref))



Matrix A:
 tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.]], device='cuda:0')
Kernel row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Reference row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 2: Column-wise Maximum

**Problem:**  

Implement a Triton kernel that computes **the maximum value in each column** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `col_max` of shape `(N,)`  
- Hint:  
  - Tile along the **M dimension** (rows)  
  - Use `tl.max` for reduction  
  - Mask out-of-bounds threads in the last tile

In [8]:
import torch
import triton
import triton.language as tl

@triton.jit
def col_max_kernel(A_ptr,
                   col_max_ptr,
                   M, N,
                   BLOCK_M: tl.constexpr):
    # Каждая программа обрабатывает один столбец
    col_idx = tl.program_id(0)
    
    # Указатель на начало текущего столбца
    # Чтобы перейти на следующую строку в том же столбце, нужно прибавить N
    col_start_ptr = A_ptr + col_idx
    
    # Инициализируем максимум очень маленьким числом
    current_max = -float('inf')
    
    # Итерируемся по строкам блоками BLOCK_M
    for off in range(0, M, BLOCK_M):
        rows = off + tl.arange(0, BLOCK_M)
        mask = rows < M
        
        # Загружаем блок элементов столбца
        # stride равен N, так как элементы одного столбца лежат через N элементов в памяти
        a_block = tl.load(col_start_ptr + rows * N, mask=mask, other=-float('inf'))
        
        # Находим максимум в блоке и обновляем глобальный максимум столбца
        block_max = tl.max(a_block, axis=0)
        current_max = tl.maximum(current_max, block_max)
        
    # Записываем финальный результат для столбца
    tl.store(col_max_ptr + col_idx, current_max)

def reference_col_max(A):
    return torch.max(A, dim=0).values


def run_col_max_kernel(A, BLOCK_M):
    # ПРОВЕРКА: BLOCK_M обязан быть степенью двойки!
    # Если передано 3, округляем до 4 для стабильности Triton
    if (BLOCK_M & (BLOCK_M - 1)) != 0:
        import math
        BLOCK_M = 2**math.ceil(math.log2(BLOCK_M))

    M, N = A.shape
    col_max = torch.zeros(N, device=A.device, dtype=A.dtype)
    grid = (N,)
    col_max_kernel[grid](A, col_max, M, N, BLOCK_M=BLOCK_M)
    return col_max

# --- Test 1: small fixed matrix ---
M, N, BLOCK_M = 6, 4, 3
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Matrix A:\n", A)
print("Kernel column max:\n", col_max)
print("Reference column max:\n", col_max_ref)
print("Match?", torch.allclose(col_max, col_max_ref))

# --- Test 2: random matrix not divisible by BLOCK_M ---
M, N, BLOCK_M = 7, 5, 3
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Random test, match?", torch.allclose(col_max, col_max_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_M = 64, 128, 16
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Large test, match?", torch.allclose(col_max, col_max_ref))

Matrix A:
 tensor([[ 1.3686,  0.7907,  0.1880,  0.7397],
        [ 0.8464, -0.7185,  0.1159, -0.1218],
        [ 0.3250,  0.3929, -0.5466, -0.8401],
        [ 0.0852, -1.0750,  0.1058,  0.7977],
        [ 1.2682,  1.9930,  0.7975, -0.7188],
        [-0.7804, -0.1767,  0.3952, -0.2212]], device='cuda:0')
Kernel column max:
 tensor([1.3686, 1.9930, 0.7975, 0.7977], device='cuda:0')
Reference column max:
 tensor([1.3686, 1.9930, 0.7975, 0.7977], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 3: GELU Activation

**Problem:**  

Implement the **GELU activation**:

$
\text{GELU}(x) = 0.5 \cdot x \cdot (1 + \tanh(\sqrt{2/\pi} \cdot (x + 0.044715 x^3)))
$

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Hint: Tile along the feature dimension and apply elementwise operations using `tl.pow`, `tl.tanh`, and broadcasting

In [10]:
import torch
import triton
import triton.language as tl
import math

@triton.jit
def gelu_kernel(X_ptr,
                Y_ptr,
                B, F,
                BLOCK_F: tl.constexpr):
    # Каждая программа обрабатывает одну строку (batch index)
    batch_idx = tl.program_id(0)
    
    # Указатель на начало текущей строки
    row_start_ptr = X_ptr + batch_idx * F
    out_start_ptr = Y_ptr + batch_idx * F
    
    # Константы для аппроксимации GELU
    SQRT_2_OVER_PI = 0.7978845608  # sqrt(2 / pi)
    COEFF = 0.044715
    
    # Итерируемся по фичам внутри строки
    for off in range(0, F, BLOCK_F):
        cols = off + tl.arange(0, BLOCK_F)
        mask = cols < F
        
        # Загружаем данные
        x = tl.load(row_start_ptr + cols, mask=mask)
        
        # Вычисляем GELU: 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
        inner = SQRT_2_OVER_PI * (x + COEFF * x * x * x)
        y = 0.5 * x * (1.0 + tl.extra.cuda.libdevice.tanh(inner))
        
        # Записываем результат
        tl.store(out_start_ptr + cols, y, mask=mask)

def reference_gelu(X):
    return torch.nn.functional.gelu(X)

def run_gelu_kernel(X, BLOCK_F):
    # Округляем до ближайшей степени двойки для Triton
    if (BLOCK_F & (BLOCK_F - 1)) != 0:
        BLOCK_F = 2**math.ceil(math.log2(BLOCK_F))
        
    B, F = X.shape
    Y = torch.empty_like(X) # Используем empty_like для скорости
    grid = (B,)
    
    gelu_kernel[grid](X, Y, B, F, BLOCK_F=BLOCK_F)
    return Y

# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 4
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("X:\n", X)
print("Kernel GELU Y:\n", Y)
print("Reference GELU Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 4
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 32
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[-0.8205,  0.5481,  1.8318, -0.5683, -0.3907,  0.1965,  0.0874, -1.4935],
        [ 0.5781, -0.1519, -2.9006, -0.7809,  0.7253,  0.5306,  2.3571, -0.3109]],
       device='cuda:0')
Kernel GELU Y:
 tensor([[-0.1691,  0.3882,  1.7704, -0.1619, -0.1360,  0.1136,  0.0467, -0.1013],
        [ 0.4153, -0.0668, -0.0050, -0.1699,  0.5554,  0.3726,  2.3358, -0.1175]],
       device='cuda:0')
Reference GELU Y:
 tensor([[-0.1690,  0.3882,  1.7704, -0.1619, -0.1360,  0.1136,  0.0467, -0.1010],
        [ 0.4153, -0.0668, -0.0054, -0.1698,  0.5555,  0.3726,  2.3354, -0.1175]],
       device='cuda:0')
Match? False
Random test, match? False
Large test, match? False


## Task 4: Layer Normalization (Row-wise)

**Problem:**  

Implement **row-wise layer normalization** (without gamma/bamma).  

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Formula:  

$
Y[i,j] = \frac{X[i,j] - \text{mean}_i}{\sqrt{\text{var}_i + \epsilon}}
$

**Hints:**  
- Compute **row mean and variance in tiles**  
- Use `tl.sum` and `tl.pow` for accumulation

In [13]:
import torch
import triton
import triton.language as tl
import math
EPS = 1e-5
@triton.jit
def layer_norm_kernel(X_ptr,
                      Y_ptr,
                      B, F,
                      BLOCK_F: tl.constexpr):
    # Каждая программа обрабатывает одну строку (batch)
    row_idx = tl.program_id(0)
    
    # Смещения для текущей строки
    row_start_ptr = X_ptr + row_idx * F
    out_start_ptr = Y_ptr + row_idx * F

    # 1. Вычисляем среднее (Mean)
    sum_val = 0.0
    for off in range(0, F, BLOCK_F):
        cols = off + tl.arange(0, BLOCK_F)
        mask = cols < F
        x = tl.load(row_start_ptr + cols, mask=mask, other=0.0)
        sum_val += tl.sum(x, axis=0)
    mean = sum_val / F

    # 2. Вычисляем дисперсию (Variance)
    var_val = 0.0
    for off in range(0, F, BLOCK_F):
        cols = off + tl.arange(0, BLOCK_F)
        mask = cols < F
        x = tl.load(row_start_ptr + cols, mask=mask, other=0.0)
        # Суммируем квадраты отклонений (центрируем данные)
        diff = tl.where(mask, x - mean, 0.0)
        var_val += tl.sum(diff * diff, axis=0)
    var = var_val / F
    
    # Константа для численной стабильности
    rstd = 1.0 / tl.sqrt(var + 1e-5)

    # 3. Финальный проход: нормализация и запись
    for off in range(0, F, BLOCK_F):
        cols = off + tl.arange(0, BLOCK_F)
        mask = cols < F
        x = tl.load(row_start_ptr + cols, mask=mask, other=0.0)
        y = (x - mean) * rstd
        tl.store(out_start_ptr + cols, y, mask=mask)

def run_layer_norm_kernel(X, BLOCK_F):
    # Гарантируем, что BLOCK_F — степень двойки
    if (BLOCK_F & (BLOCK_F - 1)) != 0:
        BLOCK_F = 2**math.ceil(math.log2(BLOCK_F))
        
    B, F = X.shape
    Y = torch.empty_like(X)
    grid = (B,)
    
    layer_norm_kernel[grid](X, Y, B, F, BLOCK_F=BLOCK_F)
    return Y

def reference_layer_norm(X, eps=EPS):
    mean = torch.mean(X, dim=1, keepdim=True)
    var = torch.var(X, dim=1, unbiased=False, keepdim=True)
    return (X - mean) / torch.sqrt(var + eps)


# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 4
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("X:\n", X)
print("Kernel LayerNorm Y:\n", Y)
print("Reference LayerNorm Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 4
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 32
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[ 1.6079,  1.7849, -1.7103, -1.2025, -0.2359, -1.0609,  1.0466, -1.6348],
        [-0.7113, -0.5047, -0.2188, -0.6104, -1.2472, -0.1350, -1.7251,  0.3416]],
       device='cuda:0')
Kernel LayerNorm Y:
 tensor([[ 1.3093,  1.4393, -1.1266, -0.7538, -0.0443, -0.6499,  0.8972, -1.0712],
        [-0.1809,  0.1590,  0.6295, -0.0149, -1.0628,  0.7674, -1.8491,  1.5517]],
       device='cuda:0')
Reference LayerNorm Y:
 tensor([[ 1.3093,  1.4393, -1.1266, -0.7538, -0.0443, -0.6499,  0.8972, -1.0712],
        [-0.1809,  0.1590,  0.6295, -0.0149, -1.0628,  0.7674, -1.8491,  1.5517]],
       device='cuda:0')
Match? True
Random test, match? True
Large test, match? True
